# 06 — Scalability and compute benchmark

Shows how many convoy updates one LinuxONE instance can score per second, for single updates and for batches up to
10,000 convoys, on one core and on all cores. Run it on LinuxONE and screenshot the summary cell as
scalability and compute evidence for the judges.

For scale: a fleet of 10,000 vehicles reporting every 10 seconds needs 1,000 Model 1 predictions per second.
Granite's speed (tokens/second) is reported in notebook 05.

Requires the artifacts from notebooks 01–03.

In [ ]:
from pathlib import Path

# Works both when Jupyter is started in the repo root and inside notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
ART = ROOT / "artifacts"
DATA.mkdir(exist_ok=True)
ART.mkdir(exist_ok=True)

import json
import os
import platform
import time
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

spec = json.loads((ART / "feature_spec.json").read_text())
CORES = os.cpu_count()

m1_rf = joblib.load(ART / "model1_rf.joblib")
m1_xgb = xgb.XGBRegressor()
m1_xgb.load_model(ART / "model1_xgb.json")
m2_rf = joblib.load(ART / "model2_rf.joblib")

X1_all = pd.read_csv(DATA / "model1_training.csv")[spec["model1_features"]]
X2_all = pd.read_csv(DATA / "model2_training.csv")[spec["model2_categorical"] + spec["model2_numeric"]]
BATCH_SIZES = [1, 100, 1_000, 10_000]
print(f"{platform.machine()} with {CORES} cores")

In [ ]:
def throughput(predict, X, min_seconds=0.5):
    """Convoys scored per second and milliseconds per call, averaged over at least min_seconds."""
    predict(X)   # warm-up
    calls, t0 = 0, time.perf_counter()
    while time.perf_counter() - t0 < min_seconds:
        predict(X)
        calls += 1
    elapsed = time.perf_counter() - t0
    return len(X) * calls / elapsed, 1000 * elapsed / calls


CONFIGS = [
    ("Model 1 Random Forest, 1 core", lambda: m1_rf.set_params(n_jobs=1), m1_rf.predict, X1_all),
    (f"Model 1 Random Forest, {CORES} cores", lambda: m1_rf.set_params(n_jobs=-1), m1_rf.predict, X1_all),
    ("Model 1 XGBoost", lambda: None, m1_xgb.predict, X1_all),
    ("Model 2 Random Forest, 1 core", lambda: m2_rf.set_params(rf__n_jobs=1), m2_rf.predict, X2_all),
    (f"Model 2 Random Forest, {CORES} cores", lambda: m2_rf.set_params(rf__n_jobs=-1), m2_rf.predict, X2_all),
]

rows = []
for label, configure, predict, X_all in CONFIGS:
    configure()
    for n in BATCH_SIZES:
        per_sec, ms = throughput(predict, X_all.iloc[:n])
        rows.append({"config": label, "convoys_per_call": n, "convoys_per_sec": per_sec, "ms_per_call": ms})
bench = pd.DataFrame(rows)
bench.pivot(index="config", columns="convoys_per_call", values="convoys_per_sec").round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for label, grp in bench.groupby("config", sort=False):
    ax.plot(grp["convoys_per_call"], grp["convoys_per_sec"], marker="o", label=label)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("convoys per call (batch size)")
ax.set_ylabel("convoys scored per second")
ax.set_title(f"Inference throughput on {platform.machine()} ({CORES} cores)")
ax.legend(fontsize=8)
plt.tight_layout()

In [ ]:
summary = {
    "machine": platform.machine(),
    "cores": CORES,
    "single_update_latency_ms": bench[bench.convoys_per_call == 1].set_index("config")["ms_per_call"].round(2).to_dict(),
    "peak_convoys_per_sec": bench.groupby("config")["convoys_per_sec"].max().round(0).to_dict(),
}
(ART / "benchmark.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))